# Notebook C — Ablation, Controls, and Quality Dose–Response (checkpointed)

Runs the protocol on both tasks. Every metadata condition is paired with two controls:

- **metadata-only** (no image) — detects leakage
- **permuted metadata** (identical architecture and parameter count, record link destroyed) — detects capacity effects

The reported quantity is **synergy**, `S = Acc(I+M) − max(Acc(I), Acc(M))`, not the naive delta.

**Checkpointing.** Each run appends a row to `runs.csv` and its test predictions to
`preds/`. On restart, completed `run_id`s are skipped. Attach a previous session's output
via `CKPT_INPUTS` to resume. The loop stops cleanly at `TIME_BUDGET_H`.

In [ ]:
import json, time, glob, warnings, itertools
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

AUD_DIR = Path('/kaggle/input/nba-audit-splits')
OUT     = Path('/kaggle/working'); OUT.mkdir(exist_ok=True, parents=True)
PRED_DIR = OUT / 'preds'; PRED_DIR.mkdir(exist_ok=True, parents=True)
CKPT_INPUTS = [
    # Path('/kaggle/input/nbc-runs-part1'),
]

SEEDS, PROTOCOLS = [0, 1, 2, 3, 4], ['spatial', 'random']
EPOCHS, PATIENCE, BATCH, LR, WD = 60, 8, 512, 1e-3, 1e-4
IMG_H, META_H, FUSE_H, DROP = 256, 64, 128, 0.3
TIME_BUDGET_H = 8.0
T0 = time.time()
elapsed_h = lambda: (time.time() - T0) / 3600
print('device:', DEVICE)

In [ ]:
# ── Load frozen artefacts ───────────────────────────────────────────────
df   = pd.read_parquet(AUD_DIR / 'corpus.parquet')
EMB  = np.load(AUD_DIR / 'resnet_emb.npy').astype(np.float32)
META = json.loads((AUD_DIR / 'splits.json').read_text())
SPL  = {k: {p: np.array(v) for p, v in s.items()} for k, s in META['splits'].items()}
assert len(df) == len(EMB)

TASKS = {'B': {'y': df.yB.values.astype(np.int64), 'classes': META['classesB'],
               'mask': np.ones(len(df), bool)},
         'A': {'y': np.where(df.yA.values >= 0, df.yA.values, 0).astype(np.int64),
               'classes': META['classesA'], 'mask': (df.yA.values >= 0)}}
print(f'{len(df):,} rows | Task A {TASKS["A"]["mask"].sum():,} | Task B {len(df):,}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  METADATA + QUALITY DEGRADATION
# ════════════════════════════════════════════════════════════════════════
@dataclass
class Q:
    precision_deg: float = 0.0
    noise_sigma:   float = 0.0
    missing_rate:  float = 0.0
    misalign_rate: float = 0.0
    permute_all:   bool  = False
    add_indicator: bool  = False
    def tag(self):
        if self.permute_all: return 'permuted'
        b = [f'{k}{v}' for k, v in [('prec', self.precision_deg), ('noise', self.noise_sigma),
             ('miss', self.missing_rate), ('misal', self.misalign_rate)] if v]
        return '+'.join(b) if b else 'clean'

CATS = {'climate_zone', 'hemisphere', 'country_code'}
VOCAB = {c: sorted(df[c].astype(str).unique()) for c in CATS if c in df.columns}

def build_meta(groups, q, train_idx, seed):
    if not groups:
        return np.zeros((len(df), 0), np.float32)
    rng = np.random.RandomState(1000 + seed)
    raw = pd.DataFrame(index=df.index)
    for g in groups:
        if g == 'gps':
            lat, lon = df.lat.values.copy(), df.lon.values.copy()
            if q.precision_deg > 0:
                lat = (np.floor(lat/q.precision_deg) + .5) * q.precision_deg
                lon = (np.floor(lon/q.precision_deg) + .5) * q.precision_deg
            raw['lat_n'] = lat/90.0; raw['lon_n'] = lon/180.0
            raw['lat_s'] = np.sin(np.radians(lat)); raw['lon_s'] = np.sin(np.radians(lon))
            raw['lon_c'] = np.cos(np.radians(lon))
        elif g == 'geo_derived':
            raw['climate_zone'] = df.climate_zone.astype(str)
            raw['hemisphere']   = df.hemisphere.astype(str)
        elif g == 'country':
            raw['country_code'] = df.country_code.astype(str)
        elif g == 'file_meta':
            for c in ['aspect', 'megapix', 'bytes_px']:
                raw[c] = df[c].astype(float)
        else:
            raise ValueError(g)

    if q.misalign_rate > 0 or q.permute_all:
        n = len(raw); k = n if q.permute_all else int(q.misalign_rate*n)
        if k > 1:
            pick = rng.choice(n, k, replace=False); sh = pick.copy(); rng.shuffle(sh)
            order = np.arange(n); order[pick] = sh
            raw = raw.iloc[order].reset_index(drop=True)

    parts = []
    for c in raw.columns:
        if c in VOCAB:
            v = VOCAB[c]; oh = np.zeros((len(raw), len(v)), np.float32)
            pos = {x: i for i, x in enumerate(v)}
            for r, x in enumerate(raw[c].astype(str).values):
                j = pos.get(x)
                if j is not None: oh[r, j] = 1.
            parts.append(oh)
        else:
            parts.append(raw[c].values.astype(np.float32).reshape(-1, 1))
    X = np.hstack(parts)

    if q.missing_rate > 0 or q.add_indicator:
        mask = rng.rand(*X.shape) < q.missing_rate
        if q.missing_rate > 0:
            fill = np.nan_to_num(np.nanmean(np.where(mask, np.nan, X)[train_idx], axis=0))
            X = np.where(mask, fill[None, :], X)
        if q.add_indicator:
            X = np.hstack([X, mask.astype(np.float32)])
    if q.noise_sigma > 0:
        sd = X[train_idx].std(0); sd[sd == 0] = 1.
        X = X + rng.randn(*X.shape).astype(np.float32) * (q.noise_sigma*sd)[None, :]
    return X.astype(np.float32)

In [ ]:
# ── Model + training ────────────────────────────────────────────────────
class Fusion(nn.Module):
    def __init__(self, di, dm, k):
        super().__init__()
        self.di, self.dm = di, dm; f = 0
        if di: self.i = nn.Sequential(nn.Linear(di, IMG_H), nn.BatchNorm1d(IMG_H),
                                      nn.ReLU(), nn.Dropout(DROP)); f += IMG_H
        if dm: self.m = nn.Sequential(nn.Linear(dm, META_H), nn.BatchNorm1d(META_H),
                                      nn.ReLU(), nn.Dropout(DROP)); f += META_H
        self.h = nn.Sequential(nn.Linear(f, FUSE_H), nn.ReLU(), nn.Dropout(DROP),
                               nn.Linear(FUSE_H, k))
    def forward(self, xi, xm):
        z = ([self.i(xi)] if self.di else []) + ([self.m(xm)] if self.dm else [])
        return self.h(torch.cat(z, 1))

def std(X, tr):
    if X.shape[1] == 0: return X
    mu, sd = X[tr].mean(0), X[tr].std(0); sd[sd == 0] = 1.
    return ((X - mu)/sd).astype(np.float32)

def run_one(task, use_img, groups, q, proto, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    T = TASKS[task]; y = T['y']; k = len(T['classes'])
    sp = {p: v[T['mask'][v]] for p, v in SPL[proto].items()}
    tr, va, te = sp['train'], sp['val'], sp['test']

    Xm = std(build_meta(groups, q, tr, seed), tr)
    Xi = std(EMB, tr) if use_img else np.zeros((len(df), 0), np.float32)
    ld = lambda ix, sh: DataLoader(TensorDataset(torch.from_numpy(Xi[ix]),
                                   torch.from_numpy(Xm[ix]), torch.from_numpy(y[ix])),
                                   batch_size=BATCH, shuffle=sh, drop_last=sh and len(ix) > BATCH)
    dtr, dva, dte = ld(tr, True), ld(va, False), ld(te, False)

    mdl = Fusion(Xi.shape[1], Xm.shape[1], k).to(DEVICE)
    cnt = np.bincount(y[tr], minlength=k).astype(np.float32)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(cnt.sum()/(k*np.maximum(cnt, 1)), device=DEVICE))
    opt = torch.optim.AdamW(mdl.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    def ev(dl):
        mdl.eval(); P, Y = [], []
        with torch.no_grad():
            for xi, xm, yy in dl:
                P.append(mdl(xi.to(DEVICE), xm.to(DEVICE)).argmax(1).cpu().numpy()); Y.append(yy.numpy())
        return np.concatenate(P), np.concatenate(Y)

    best, bad, state, ep = -1, 0, None, 0
    for ep in range(EPOCHS):
        mdl.train()
        for xi, xm, yy in dtr:
            opt.zero_grad(); crit(mdl(xi.to(DEVICE), xm.to(DEVICE)), yy.to(DEVICE)).backward(); opt.step()
        sch.step()
        pv, yv = ev(dva); f = f1_score(yv, pv, average='macro')
        if f > best + 1e-5:
            best, bad = f, 0; state = {a: b.detach().clone() for a, b in mdl.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    mdl.load_state_dict(state)
    pt, yt = ev(dte)
    return dict(test_acc=accuracy_score(yt, pt), test_macro_f1=f1_score(yt, pt, average='macro'),
                test_bal_acc=balanced_accuracy_score(yt, pt), val_macro_f1=best,
                epochs=ep+1, meta_dim=int(Xm.shape[1]),
                n_params=sum(p.numel() for p in mdl.parameters())), pt.astype(np.int16), yt.astype(np.int16)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  EXPERIMENT REGISTRY
# ════════════════════════════════════════════════════════════════════════
PRIMARY = ['gps', 'geo_derived']
COND = [
    ('C0', 'Image only',              True,  [],                      Q(),  'ablation'),
    ('C1', 'Image + file metadata',   True,  ['file_meta'],           Q(),  'ablation'),
    ('C2', 'Image + climate/hemis',   True,  ['geo_derived'],         Q(),  'ablation'),
    ('C3', 'Image + GPS 5deg',        True,  ['gps'],   Q(precision_deg=5.0), 'ablation'),
    ('C4', 'Image + GPS exact',       True,  ['gps'],                 Q(),  'ablation'),
    ('C5', 'Image + GPS + climate',   True,  PRIMARY,                 Q(),  'ablation'),
    ('C6', 'Image + all metadata',    True,  PRIMARY+['file_meta'],   Q(),  'ablation'),
    ('O1', 'ORACLE Image + country',  True,  ['country'],             Q(),  'oracle'),
    ('M0', 'Metadata only: GPS',      False, ['gps'],                 Q(),  'control'),
    ('M1', 'Metadata only: primary',  False, PRIMARY,                 Q(),  'control'),
    ('M2', 'Metadata only: file',     False, ['file_meta'],           Q(),  'control'),
    ('M3', 'Metadata only: country',  False, ['country'],             Q(),  'control'),
    ('P0', 'Image + PERMUTED GPS',    True,  ['gps'],   Q(permute_all=True), 'control'),
    ('P1', 'Image + PERMUTED primary',True,  PRIMARY,   Q(permute_all=True), 'control'),
]
GRID = []
for p in [0.0, 0.25, 0.5, 0.75, 1.0]:
    GRID.append((f'Qmiss{p}', PRIMARY, Q(missing_rate=p, add_indicator=True), 'missing', p))
for s in [0.0, 0.25, 1.0, 2.0, 4.0]:
    GRID.append((f'Qnoise{s}', PRIMARY, Q(noise_sigma=s), 'noise', s))
for m in [0.0, 0.25, 0.5, 0.75, 1.0]:
    GRID.append((f'Qmisal{m}', PRIMARY, Q(misalign_rate=m), 'misalign', m))
for d in [0.01, 0.5, 5.0, 20.0, 60.0]:
    GRID.append((f'Qprec{d}', ['gps'], Q(precision_deg=d), 'precision', d))

JOBS = []
for t in ['B', 'A']:
    for cid, lab, ui, g, q, fam in COND:
        for pr in PROTOCOLS:
            for s in SEEDS:
                JOBS.append(dict(run_id=f'{t}|{cid}|{pr}|s{s}', task=t, condition=cid, label=lab,
                                 family=fam, protocol=pr, seed=s, use_image=ui, groups=g, q=q,
                                 axis='', level=np.nan))
for cid, g, q, ax, lv in GRID:                      # dose-response: Task B, spatial only
    for s in SEEDS:
        JOBS.append(dict(run_id=f'B|{cid}|spatial|s{s}', task='B', condition=cid,
                         label=f'{ax}={lv}', family='quality', protocol='spatial', seed=s,
                         use_image=True, groups=g, q=q, axis=ax, level=lv))
print(f'{len(JOBS)} total runs planned')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  RUN with checkpoint / resume
# ════════════════════════════════════════════════════════════════════════
def load_done():
    frames, seen = [], set()
    for d in [OUT] + [Path(p) for p in CKPT_INPUTS]:
        f = Path(d) / 'runs.csv'
        if f.exists():
            r = pd.read_csv(f); frames.append(r); seen |= set(r.run_id)
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), seen

done_df, done = load_done()
print(f'{len(done)} runs already complete, {len(JOBS)-len(done)} remaining')

rows = []
for n, j in enumerate(JOBS):
    if j['run_id'] in done:
        continue
    if elapsed_h() > TIME_BUDGET_H:
        print(f'\nTIME BUDGET reached. Save a version, publish output, add to CKPT_INPUTS, re-run.')
        break
    m, pt, yt = run_one(j['task'], j['use_image'], j['groups'], j['q'], j['protocol'], j['seed'])
    np.savez_compressed(PRED_DIR / f"{j['run_id'].replace('|','_')}.npz", pred=pt, label=yt)
    rows.append({**{k: v for k, v in j.items() if k not in ('groups', 'q', 'use_image')},
                 'groups': '+'.join(j['groups']) or 'none', 'quality': j['q'].tag(),
                 'use_image': j['use_image'], **m})
    if len(rows) % 5 == 0 or n == len(JOBS)-1:
        pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).to_csv(OUT/'runs.csv', index=False)
        print(f"[{n+1}/{len(JOBS)}] {j['run_id']:28s} acc={m['test_acc']:.4f} "
              f"f1={m['test_macro_f1']:.4f} | {elapsed_h():.2f}h", flush=True)

runs = pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True)
runs.to_csv(OUT / 'runs.csv', index=False)
json.dump({'seeds': SEEDS, 'protocols': PROTOCOLS, 'epochs': EPOCHS, 'batch': BATCH,
           'lr': LR, 'wd': WD, 'img_h': IMG_H, 'meta_h': META_H, 'fuse_h': FUSE_H,
           'dropout': DROP, 'primary': PRIMARY, 'classesA': META['classesA'],
           'classesB': META['classesB'], 'majorityA': META['majorityA'],
           'majorityB': META['majorityB'], 'n_runs': int(len(runs))},
          open(OUT / 'experiment_config.json', 'w'), indent=2)
print(f'\n{len(runs)}/{len(JOBS)} runs complete')


In [ ]:
# ── Quick read-out ──────────────────────────────────────────────────────
for t in ['B', 'A']:
    s = runs.query('task == @t and protocol == "spatial"').groupby('condition').test_acc.mean()
    if s.empty: continue
    print(f'\n--- Task {t} (spatial) ---')
    for c in ['C0', 'C4', 'C5', 'M0', 'M1', 'P0', 'P1', 'O1']:
        if c in s: print(f'  {c:4s} {s[c]:.4f}  (vs image-only {s[c]-s.get("C0", np.nan):+.4f})')
print('\nExpected: Task A -> M0 ~ ceiling, synergy ~ 0.  Task B -> C5 > max(C0, M1).')